In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofweek, regexp_replace
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import os

In [0]:
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/previsao_vendas_shoes_dev/source_csv/modelo_ml"

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
df_fato_vendas = spark.read.table("previsao_vendas_shoes_dev.ecommerce.fato_vendas")

In [0]:
df_fato_vendas_com_colunas_adicionadas = (
    df_fato_vendas.withColumn("mes", month(col("data_venda")))
    .withColumn("ano", year(col("data_venda")))
    .withColumn("dia_semana", dayofweek(col("data_venda")))
    .withColumn("vlr_pedido", regexp_replace(col("vlr_pedido"), "R\\$|\\s", ""))
    .withColumn("vlr_pedido", regexp_replace(col("vlr_pedido"), ",", ".")) 
    .withColumn("vlr_pedido", col("vlr_pedido").cast("double")) 
    .withColumn("vlr_venda", col("qtd_vendas") * col("vlr_pedido"))
      
)
feature_cols = ["ano", "mes", "dia_semana", "qtd_vendas", "vlr_pedido"]
target_col = "vlr_venda"

In [0]:
assemble = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assemble.transform(df_fato_vendas_com_colunas_adicionadas).select("features", target_col)

In [0]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

In [0]:

lr = LinearRegression(featuresCol="features", labelCol=target_col)
model = lr.fit(train_df)

In [0]:

predictions = model.transform(test_df)
predictions.select("features", target_col, "prediction").show(10)

In [0]:

evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse}")

In [0]:

mae = RegressionEvaluator(labelCol="vlr_venda", predictionCol="prediction", metricName="mae").evaluate(predictions)
print(f"MAE: {mae}")

In [0]:

r2 = RegressionEvaluator(labelCol="vlr_venda", predictionCol="prediction", metricName="r2").evaluate(predictions)
print(f"R²: {r2}")

In [0]:


if mlflow.active_run():
    mlflow.end_run()
else:
    mlflow.start_run()
    mlflow.spark.log_model(model, "modelo_previsao_vendas", pip_requirements=["pyspark==4.0.0", "mlflow"])
    mlflow.log_metric("rmse", rmse)
    mlflow.end_run()